# Fine-tune and evaluate the log pipeline (Colab)

Run **Setup** first, then any section on its own:

1. **DeepParse fine-tune**: build the training set (Loghub-2k + synthetic IoT_fleet) and LoRA-train the mask model.
2. **DeepParse eval**: GA/PA per system, offline masks vs. the fine-tuned adapter.
3. **logpipe evals**: parsing, alias, IoT fleet end to end, LogBERT on HDFS/BGL, then `REPORT.md`.

Runtime: GPU (T4 works; L4/A100 is much faster for the 8B model). Checkpoints and results go to Google Drive, so a disconnect only costs the current step.

## Setup

In [ ]:
REPO = "https://github.com/Niang931/Log_parser.git"  #@param {type:"string"}
BRANCH = "feat/finetune-deeparse-iot"  #@param {type:"string"}
TRAIN_CONFIG = "configs/train_iot.yaml"  #@param ["configs/train_iot.yaml", "configs/train_iot_qwen3b.yaml"]
DRIVE_DIR = "/content/drive/MyDrive/logparser"  #@param {type:"string"}
LOGBERT_EPOCHS = 30  #@param {type:"integer"}

In [ ]:
import os, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

# private repo: add a Colab secret called GITHUB_TOKEN
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO.replace("https://", f"https://{token}@") if token else REPO

ROOT = "/content/Log_parser"
if not os.path.exists(ROOT):
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, ROOT], capture_output=True)
    print("clone ok" if r.returncode == 0 else "clone failed (check REPO, BRANCH, GITHUB_TOKEN)")
DP = f"{ROOT}/DeepParse"
CKPT = f"{DRIVE_DIR}/checkpoints/" + os.path.basename(TRAIN_CONFIG).removesuffix(".yaml")
EVAL_OUT = f"{DRIVE_DIR}/eval"

In [ ]:
# the repo asks for Python 3.13; Colab's older Python works for this code
!pip install -q -e "{DP}[train]" bitsandbytes
!pip install -q --ignore-requires-python -e {ROOT}
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. DeepParse fine-tune

Data: the 14 Loghub-2k systems (50 lines each, entropy-greedy, as in the paper) plus the synthetic IoT_fleet (4 lines per template so rare value shapes are seen several times).

In [ ]:
%cd {DP}
!python -m deepparse.tools.fetch_loghub --out artifacts/data
!python -m deepparse.tools.synth_iot_fleet --out-dir artifacts/data/IoT_fleet

In [ ]:
from deepparse.tools.fetch_loghub import DEFAULT_SYSTEMS
SYSTEMS = " ".join(DEFAULT_SYSTEMS + ["IoT_fleet"])

!python -m deepparse.tools.build_training_set --systems {SYSTEMS} \
    --entropy-k 50 --system-per-template IoT_fleet=4 \
    --out artifacts/training/train_combined.jsonl
!wc -l artifacts/training/train_combined.jsonl

Training resumes from the last checkpoint on Drive if the runtime dropped. Quick smoke test first? Add `--max-steps 3`.

In [ ]:
!python -m deepparse.training.finetune --config {TRAIN_CONFIG} --output-dir {CKPT} --resume

## 2. DeepParse eval

Same 15 systems twice: first with the offline (rule-based) masks as the baseline, then with masks written by the fine-tuned adapter. Saved masks are reused by `deepparse eval`, so each run starts from an empty `artifacts/masks`.

In [ ]:
%cd {DP}
!mkdir -p {EVAL_OUT}/deepparse
!rm -f artifacts/masks/*.json
!deepparse eval --config configs/eval_with_iot.yaml --deterministic
!cp artifacts/outputs/eval_with_iot_metrics.csv {EVAL_OUT}/deepparse/offline.csv

One model load per system, so this takes a while for the 8B model.

In [ ]:
!rm -f artifacts/masks/*.json
!bash scripts/synth_all_with_adapter.sh {CKPT}
!deepparse eval --config configs/eval_with_iot.yaml --deterministic
!cp artifacts/outputs/eval_with_iot_metrics.csv {EVAL_OUT}/deepparse/finetuned.csv
!cp -r artifacts/masks {EVAL_OUT}/deepparse/masks_finetuned

In [ ]:
import pandas as pd

off = pd.read_csv(f"{EVAL_OUT}/deepparse/offline.csv").set_index("dataset")[["GA", "PA"]]
ft = pd.read_csv(f"{EVAL_OUT}/deepparse/finetuned.csv").set_index("dataset")[["GA", "PA"]]
cmp = off.join(ft, lsuffix=" offline", rsuffix=" finetuned")
cmp["GA diff"] = cmp["GA finetuned"] - cmp["GA offline"]
cmp["PA diff"] = cmp["PA finetuned"] - cmp["PA offline"]
cmp.round(4)

## 3. logpipe evals

Each cell writes into `EVAL_OUT`; the last one builds `REPORT.md` with an explanation of every table. These use the offline LLM adapter (no API key); pass `--llm llm:anthropic` etc. to use a hosted one.

In [ ]:
%cd {ROOT}
!logpipe eval parsing --out {EVAL_OUT}
!logpipe eval parsing --no-tune --out {EVAL_OUT}/parsing_untuned
!logpipe eval alias --out {EVAL_OUT}

In [ ]:
!logpipe eval iot --epochs {LOGBERT_EPOCHS} --out {EVAL_OUT}

In [ ]:
!logpipe fetch-data HDFS BGL
!logpipe eval logbert --dataset HDFS --epochs {LOGBERT_EPOCHS} --out {EVAL_OUT}
!logpipe eval logbert --dataset BGL --epochs {LOGBERT_EPOCHS} --out {EVAL_OUT}

In [ ]:
from IPython.display import Markdown
!logpipe eval report --out {EVAL_OUT}
Markdown(open(f"{EVAL_OUT}/REPORT.md").read())